In [1]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans

df = pd.read_csv("../data/processed/dataset_final.csv")

# criteria: the 4 factors, ALL benefit (costs already inverted upstream)
benefit_cols = ["Demand", "Accessibility", "Competition", "Population"]
cost_cols    = []
criteria = benefit_cols + cost_cols

# min-max normalization
normalized = pd.DataFrame(index=df.index)
for col in benefit_cols:
    mn, mx = df[col].min(), df[col].max()
    normalized[col] = 1 if mx == mn else (df[col]-mn)/(mx-mn)
for col in cost_cols:
    mn, mx = df[col].min(), df[col].max()
    normalized[col] = 1 if mx == mn else (mx-df[col])/(mx-mn)

# entropy weight method
eps = 1e-12
P = normalized.div(normalized.sum(axis=0), axis=1).replace(0, eps)
n = len(df)
entropy    = -(P*np.log(P)).sum(axis=0)/np.log(n)
divergence = 1 - entropy
weights    = divergence/divergence.sum()

weight_table = pd.DataFrame({
    "feature": criteria,
    "criterion_type": ["Benefit"]*len(benefit_cols)+["Cost"]*len(cost_cols),
    "entropy": entropy.values, "divergence": divergence.values, "weight": weights.values
}).sort_values("weight", ascending=False)
weight_table.to_csv("../data/processed/entropy_feasibility_weightage.csv", index=False)

# feasibility score
for col in criteria:
    df[col+"_contribution"] = normalized[col]*weights[col]
df["feasibility_score"] = df[[c+"_contribution" for c in criteria]].sum(axis=1)*100

# KMeans tiers
km = KMeans(n_clusters=3, random_state=42, n_init=20)
df["cluster"] = km.fit_predict(df[["feasibility_score"]])
order = df.groupby("cluster")["feasibility_score"].mean().sort_values().index
df["feasibility_label"] = df["cluster"].map({order[0]:"Low", order[1]:"Moderate", order[2]:"High"})

df.to_csv("../data/processed/dataset_final_entropy.csv", index=False)
print(weight_table.to_string(index=False))
print(df["feasibility_label"].value_counts())

      feature criterion_type  entropy  divergence   weight
       Demand        Benefit 0.974984    0.025016 0.418212
   Population        Benefit 0.980038    0.019962 0.333723
  Competition        Benefit 0.990422    0.009578 0.160128
Accessibility        Benefit 0.994740    0.005260 0.087936
feasibility_label
Moderate    1692
High        1378
Low         1102
Name: count, dtype: int64
